In [1]:
import os, sys
import time
from datetime import datetime
from pathlib import Path
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from dataset import DTIDataset
from model import DeepDTAF, DeepDTAF_3to2, DeepDTAF_2to1, test, DTIDenseNet3to1, DTIDenseNet3to2, DTIDenseNet2to1, Loss_fc
import sklearn.metrics as m
from scipy.stats import pearsonr
from numba import njit
import xlwt

In [2]:
@njit
def c_index(y_true, y_pred):
    summ = 0
    pair = 0

    for i in range(1, len(y_true)):
        for j in range(0, i):
            pair += 1
            if y_true[i] > y_true[j]:
                summ += 1 * (y_pred[i] > y_pred[j]) + 0.5 * (y_pred[i] == y_pred[j])
            elif y_true[i] < y_true[j]:
                summ += 1 * (y_pred[i] < y_pred[j]) + 0.5 * (y_pred[i] == y_pred[j])
            else:
                pair -= 1

    if pair != 0:
        return summ / pair
    else:
        return 0

def MSE(y_true, y_pred):
    values = np.square(y_true - y_pred)
    return np.mean(values), np.std(values)

def MAE(y_true, y_pred):
    values = np.abs(y_true - y_pred)
    return np.mean(values), np.std(values)

def CORR(y_true, y_pred):
    return pearsonr(y_true, y_pred)[0]

def SD(y_true, y_pred):
    from sklearn.linear_model import LinearRegression
    y_pred = y_pred.reshape((-1,1))
    lr = LinearRegression().fit(y_pred,y_true)
    y_ = lr.predict(y_pred)
    return np.sqrt(np.square(y_true - y_).sum() / (len(y_pred) - 1))

def save_xlt(path, names, targets, outputs):
    
    wbk = xlwt.Workbook()
    sheet = wbk.add_sheet('Sheet1',cell_overwrite_ok=True)
    sheet.write(0,0, 'id')
    sheet.write(0,1, 'target')
    sheet.write(0,2, 'pred_full')
    sheet.write(0,3, 'pred_only_seq')
    sheet.write(0,4, 'pred_only_pkt')
    
    for idx,(i,t,p,p_os,p_op) in enumerate(zip(names, targets, outputs[0], outputs[1], outputs[2])):
        sheet.write(idx+1,0, i)   
        sheet.write(idx+1,1, float(t))
        sheet.write(idx+1,2, float(p))
        sheet.write(idx+1,3, float(p_os))
        sheet.write(idx+1,4, float(p_op))

    wbk.save(path)
    
name_list, target_list, output_list, output_os_list, output_op_list = [], [], [], [], []

In [3]:
data_path, mode, settype = '../DTI-dataset', '3to1_050100_r20', 'test'
device = torch.device("cuda:1")  # or torch.device('cpu')

onehot = False
dilation = True  # 使用扩张卷积
batch_size = 256

# 大部分情况下，设置这个 flag 可以让内置的 cuDNN 的 auto-tuner 自动寻找最适合当前配置的高效算法，来达到优化运行效率的问题。
max_seq_len = 904  # 数据预处理使用的值，定义序列长度，蛋白
max_pkt_len = 64  # 口袋
max_smi_len = 154  # 化合物
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

for fold in range(10):
    fold +=1
    print(f'processing {fold}/10')
    
#     model = DTIDenseNet3to2(onehot=onehot, dilation=dilation)
    model = DTIDenseNet3to1(onehot=onehot, dilation=dilation)
#     model = DTIDenseNet2to1(onehot=onehot, dilation=dilation)
#     model = DeepDTAF_3to2()
#     model = DeepDTAF_2to1()
#     model = DeepDTAF()

    for l in os.listdir(f'/data1/zxy/DTI Results/Dense433_3to1-drop_p_test/{mode}/'):
        if f'fold{fold}_' in l:
            path = Path(os.path.join(f'/data1/zxy/DTI Results/Dense433_3to1-drop_p_test/{mode}/', l))
            print(path)
    model.load_state_dict(torch.load(path / 'best_model.pt'))
    model = model.to(device)
    model.eval()
    
    data_loaders = {'training':
                        DataLoader(DTIDataset(data_path, f'fold_{fold}/training',
                                              max_seq_len, max_pkt_len, max_smi_len, 
                                              pkt_window=None, pkt_stride=None, 
                                              onehot=onehot),
                                   batch_size=batch_size,
                                   pin_memory=True,
                                   num_workers=4,
                                   shuffle=False),
                    'validation':
                        DataLoader(DTIDataset(data_path, f'fold_{fold}/validation',
                                              max_seq_len, max_pkt_len, max_smi_len, 
                                              pkt_window=None, pkt_stride=None, 
                                              onehot=onehot),
                                   batch_size=batch_size,
                                   pin_memory=True,
                                   num_workers=4,
                                   shuffle=False),
                    'test':
                        DataLoader(DTIDataset(data_path, f'fold_{fold}/test',
                                              max_seq_len, max_pkt_len, max_smi_len, 
                                              pkt_window=None, pkt_stride=None, 
                                              onehot=onehot),
                                   batch_size=batch_size,
                                   pin_memory=True,
                                   num_workers=4,
                                   shuffle=False)
                   }

    outputs, outputs_os, outputs_op = [], [], []
    targets = []
    names = []
    with torch.no_grad():
        for idx, (*x, y, name) in enumerate(data_loaders[settype]):
            seq, pkt, smi = x[0].to(device), x[1].to(device), x[2].to(device)
            seq_dead = torch.zeros_like(seq)
            pkt_dead = torch.zeros_like(pkt)
            y = y.to(device)

            if isinstance(model, DTIDenseNet3to2) or isinstance(model, DeepDTAF_3to2):
                y_hat_only_seq, y_hat_only_pkt = model(seq, pkt, smi)
                y_hat_full = (y_hat_only_seq+y_hat_only_pkt)/2

            else:
                y_hat_full = model(seq, pkt, smi)
                y_hat_only_seq = model(seq, pkt_dead, smi)
                y_hat_only_pkt = model(seq_dead, pkt, smi)

            outputs.append(y_hat_full.cpu().numpy().reshape(-1))
            outputs_os.append(y_hat_only_seq.cpu().numpy().reshape(-1))
            outputs_op.append(y_hat_only_pkt.cpu().numpy().reshape(-1))
            targets.append(y.cpu().numpy().reshape(-1))
            names += name

    # output and comparison
    targets = np.concatenate(targets).reshape(-1)
    outputs = np.concatenate(outputs).reshape(-1)
    outputs_os = np.concatenate(outputs_os).reshape(-1)
    outputs_op = np.concatenate(outputs_op).reshape(-1)

#     evaluation_full = {
#         'c_index': c_index(targets, outputs),
#         'MSE': MSE(targets, outputs),
#         'MAE': MAE(targets, outputs),
#         'SD': SD(targets, outputs),
#         'CORR': CORR(targets, outputs)
#     }
#     print(f'fold{fold}_{settype}_evaluation_full:')
#     for k, v in evaluation_full.items():
#     #     print(f'{k}: {v:.3f}')
#         print(f'{k}: {v}')

#     evaluation_only_seq = {
#         'c_index': c_index(targets, outputs_os),
#         'MSE': MSE(targets, outputs_os),
#         'MAE': MAE(targets, outputs_os),
#         'SD': SD(targets, outputs_os),
#         'CORR': CORR(targets, outputs_os)
#     }
#     print('\n'+f'fold{fold}_{settype}_evaluation_only_seq:')
#     for k, v in evaluation_only_seq.items():
#     #     print(f'{k}: {v:.3f}')
#         print(f'{k}: {v}')

#     evaluation_only_pkt = {
#         'c_index': c_index(targets, outputs_op),
#         'MSE': MSE(targets, outputs_op),
#         'MAE': MAE(targets, outputs_op),
#         'SD': SD(targets, outputs_op),
#         'CORR': CORR(targets, outputs_op)
#     }
#     print('\n'+f'fold{fold}_{settype}_evaluation_only_pkt:')
#     for k, v in evaluation_only_pkt.items():
#     #     print(f'{k}: {v:.3f}')
#         print(f'{k}: {v}')

    print(path/f'{settype}_compare.xls')
    save_xlt(path=path/f'{settype}_compare.xls', names=names, targets=targets, outputs=[outputs, outputs_os, outputs_op])

    target_list.append(targets)
    output_list.append(outputs)
    output_os_list.append(outputs_os)
    output_op_list.append(outputs_op)
    name_list += names
    

processing 1/10
/data1/zxy/DTI Results/Dense433_3to1-drop_p_test/3to1_050100_r20/3to1_drop_r20_fold1_20220418225421_18885
Dataset fold_1/training: will not fold pkt
Dataset fold_1/validation: will not fold pkt
Dataset fold_1/test: will not fold pkt
/data1/zxy/DTI Results/Dense433_3to1-drop_p_test/3to1_050100_r20/3to1_drop_r20_fold1_20220418225421_18885/test_compare.xls
processing 2/10
/data1/zxy/DTI Results/Dense433_3to1-drop_p_test/3to1_050100_r20/3to1_drop_r20_fold2_20220421110819_18885
Dataset fold_2/training: will not fold pkt
Dataset fold_2/validation: will not fold pkt
Dataset fold_2/test: will not fold pkt
/data1/zxy/DTI Results/Dense433_3to1-drop_p_test/3to1_050100_r20/3to1_drop_r20_fold2_20220421110819_18885/test_compare.xls
processing 3/10
/data1/zxy/DTI Results/Dense433_3to1-drop_p_test/3to1_050100_r20/3to1_drop_r20_fold3_20220421110846_18885
Dataset fold_3/training: will not fold pkt
Dataset fold_3/validation: will not fold pkt
Dataset fold_3/test: will not fold pkt
/data1/

In [4]:
targets_all = np.concatenate(target_list).reshape(-1)
outputs_all = np.concatenate(output_list).reshape(-1)
outputs_os_all = np.concatenate(output_os_list).reshape(-1)
outputs_op_all = np.concatenate(output_op_list).reshape(-1)
print(len(targets_all))

print('MAE:', MAE(targets_all, outputs_all)[0], MAE(targets_all, outputs_all)[1])
print('MSE:', MSE(targets_all, outputs_all)[0], MSE(targets_all, outputs_all)[1])
print('SD:', SD(targets_all, outputs_all))
print('CORR:', CORR(targets_all, outputs_all))
print('c_index:', c_index(targets_all, outputs_all))

save_xlt(path=os.path.join(f'/data1/zxy/DTI Results/Dense433_3to1-drop_p_test/{mode}/', f'{settype}_compare.xls'), \
         names=name_list, targets=targets_all, outputs=[outputs_all, outputs_os_all, outputs_op_all])

18990
MAE: 0.93047017 0.8018473
MSE: 1.5087339 2.8762486
SD: 1.219832615004485
CORR: 0.7510876623226806
c_index: 0.7784870770585527
